In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from model.model_cifar10 import CIFAR10Classifier
import torch.nn.functional as F

In [13]:
# 1. Data Preparation
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),  # Randomly flip the image horizontally
    transforms.RandomCrop(32, padding=4),  # Randomly crop
    transforms.ToTensor(),  # Convert to tensor
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))  # Normalize using CIFAR-10 stats
])

batch_size = 64
train_dataset = datasets.CIFAR10(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.CIFAR10(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

Files already downloaded and verified
Files already downloaded and verified


In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CIFAR10Classifier().to(device)

# 3. Loss Function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [15]:
# 4. Training Loop with Learning Rate Scheduler
def train_model(model, train_loader, criterion, optimizer, device, num_epochs=10):
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)  # Decrease LR by a factor of 0.1 every 5 epochs
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
            # Calculate accuracy
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
        # Step the scheduler
        scheduler.step()
        
        epoch_loss = running_loss / len(train_loader)
        epoch_accuracy = 100 * correct / total
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.2f}%, LR: {scheduler.get_last_lr()[0]:.6f}")


In [16]:
train_model(model, train_loader, criterion, optimizer, device, num_epochs=30)

Epoch [1/15], Loss: 1.4071, Accuracy: 49.36%
Epoch [2/15], Loss: 1.0723, Accuracy: 61.94%
Epoch [3/15], Loss: 0.9529, Accuracy: 66.46%
Epoch [4/15], Loss: 0.8711, Accuracy: 69.58%
Epoch [5/15], Loss: 0.8209, Accuracy: 71.35%
Epoch [6/15], Loss: 0.7720, Accuracy: 72.95%
Epoch [7/15], Loss: 0.7333, Accuracy: 74.38%
Epoch [8/15], Loss: 0.7029, Accuracy: 75.10%
Epoch [9/15], Loss: 0.6693, Accuracy: 76.51%
Epoch [10/15], Loss: 0.6470, Accuracy: 77.38%
Epoch [11/15], Loss: 0.6166, Accuracy: 78.33%
Epoch [12/15], Loss: 0.5979, Accuracy: 78.80%
Epoch [13/15], Loss: 0.5789, Accuracy: 79.41%
Epoch [14/15], Loss: 0.5597, Accuracy: 80.16%
Epoch [15/15], Loss: 0.5361, Accuracy: 80.94%


In [19]:
# 5. Evaluation
def evaluate_model(model, test_loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    print(f"Test Accuracy: {100 * correct / total:.2f}%")

# Evaluate the model
evaluate_model(model, test_loader, device)

# 6. Save the model
torch.save(model.state_dict(), "model/cifar10_model.pth")

Test Accuracy: 75.97%
